# P3 - Annotation Prosodique pour TTS Agentique

**Navigation** : [Index](../README.md) | [<< Precedent](04-9-Voice-Casting.ipynb) | [Suivant >>](04-11-Generation-TTS.ipynb)

**Epic #1028** | Pipeline 5-pass : Lecture analytique > Voice casting > **Annotation prosodique** > Generation TTS > Compilation audio

Ce notebook enrichit les repliques identifiees en P1 avec des tags expressifs FishAudio S2-Pro. FishAudio supporte 15000+ tags expressifs (joie, colere, chuchotement, soupir, etc.) qui permettent un controle fin de la prosodie pour chaque personnage.

## Objectifs

1. Charger les repliques (P1) et le voice casting (P2)
2. Analyser le contexte emotionnel de chaque replique
3. Assigner des tags expressifs FishAudio S2-Pro adaptes
4. Exporter le resultat en markdown enrichi (`book.ssml.md`)

**Duree estimee** : 8-10 minutes

---


## 1. Configuration et imports

In [1]:
import json
import csv
import re
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional

BASE_DIR = Path("output_analytique")
VOICE_CONFIG = Path("voice_casting_output/voice_casting_config.json")
OUTPUT_DIR = Path("prosodic_output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"P1 output dir: {BASE_DIR.resolve()}")
print(f"P2 voice config: {VOICE_CONFIG.resolve()}")
print(f"P3 output dir: {OUTPUT_DIR.resolve()}")

P1 output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\output_analytique
P2 voice config: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\voice_casting_output\voice_casting_config.json
P3 output dir: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\prosodic_output


## 2. Tags expressifs FishAudio S2-Pro

FishAudio S2-Pro supporte des tags expressifs inseres directement dans le texte entre crochets. Contrairement au SSML standard (limites a `<break>`, `<emphasis>`, `<prosody>`), les tags FishAudio sont beaucoup plus riches :

| Categorie | Tags exemples | Usage narratif |
|-----------|---------------|----------------|
| **Emotions** | `[laugh]`, `[sob]`, `[sigh]`, `[gasp]` | Reactions spontanees |
| **Intensite vocale** | `[whisper]`, `[shout]`, `[mutter]` | Volume et projection |
| **Tonalite** | `[cold]`, `[warm]`, `[sarcastic]`, `[tender]` | Attitude du personnage |
| **Rythme** | `[pause]`, `[quick]`, `[slow]`, `[hesitate]` | Cadence de la parole |
| **Respiration** | `[inhale]`, `[exhale]`, `[choke]` | Effects physiologiques |
| **Paralinguistique** | `[yawn]`, `[cough]`, `[sniff]`, `[giggle]` | Sons non-verbaux |

La strategie d'annotation combine analyse du texte (mots-cles, ponctuation) et contexte narratif (role du personnage, type de segment).

In [2]:
# FishAudio S2-Pro expressive tag taxonomy
# Organized by category for programmatic annotation

FISHAUDIO_TAGS = {
    "emotions": {
        "laugh": ["rire", "sourire", "sourit", "riant", "gaiete"],
        "sob": ["pleure", "pleurant", "larmes", "sanglot"],
        "sigh": ["soupir", "soupira", "soupirant"],
        "gasp": ["stupefait", "choque", "horripile", "effraye", "hoquet"],
        "groan": ["gemissement", "gemit", "grogne"],
    },
    "intensity": {
        "whisper": ["murmura", "murmure", "chuchota", "chuchotement", "voix basse", "a voix basse"],
        "shout": ["cria", "crier", "hurlement", "vocifera", "hurla"],
        "mutter": ["marmonna", "grommela", "bougonna", "entre ses dents"],
    },
    "tone": {
        "cold": ["froidement", "froid", "glacial", "sans emotion", "indifference"],
        "warm": ["chaleureux", "tendrement", "avec douceur", "affectueux"],
        "sarcastic": ["sarcastique", "ironique", "ironiquement", "derision"],
        "tender": ["tendresse", "tendrement", "avec tendresse", "doux"],
        "indignant": ["indignation", "indigne", "furieux", "colere", "courroux"],
        "timid": ["timidement", "timide", "hesitant", "craintivement"],
        "onctuous": ["onction", "onctueusement", "sousveillance", "diplomatie", "avec un sourire"],
    },
    "rhythm": {
        "pause": ["...", " - ", "silence", "un moment"],
        "hesitate": ["euh", "bah", "enfin", "comment dire"],
        "quick": ["precipitamment", "vite", "hative", "preSSIPit"],
    },
    "breath": {
        "inhale": ["inspira", "aspiration", "reprenant son souffle"],
        "exhale": ["expira", "souffla", "relachement"],
        "choke": ["etouffa", "etrangla", "voix tremblante"],
    },
}

print(f"Taxonomie FishAudio : {sum(len(v) for v in FISHAUDIO_TAGS.values())} tags en {len(FISHAUDIO_TAGS)} categories")
for cat, tags in FISHAUDIO_TAGS.items():
    print(f"  {cat}: {', '.join(tags.keys())}")

Taxonomie FishAudio : 21 tags en 5 categories
  emotions: laugh, sob, sigh, gasp, groan
  intensity: whisper, shout, mutter
  tone: cold, warm, sarcastic, tender, indignant, timid, onctuous
  rhythm: pause, hesitate, quick
  breath: inhale, exhale, choke


## 3. Chargement des donnees P1 et P2

In [3]:
def load_p1_data() -> tuple:
    # Load P1 analysis results
    with open(BASE_DIR / "analyse_boule_de_suif.json", encoding="utf-8") as f:
        analysis = json.load(f)

    # Load utterances (repliques)
    utterances = []
    with open(BASE_DIR / "repliques.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            utterances.append(row)

    # Load character info
    characters = []
    with open(BASE_DIR / "personnages.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            characters.append(row)

    # Load segments
    segments = []
    with open(BASE_DIR / "segments.csv", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            segments.append(row)

    return analysis, utterances, characters, segments


def load_p2_config() -> dict:
    # Load P2 voice casting configuration
    with open(VOICE_CONFIG, encoding="utf-8") as f:
        return json.load(f)


analysis, utterances, characters, segments = load_p1_data()
voice_config = load_p2_config()

print(f"P1 - Segments: {len(segments)}, Repliques: {len(utterances)}, Personnages: {len(characters)}")
print(f"P2 - Personnages castes: {len(voice_config['characters'])}")
print(f"\nPremieres repliques:")
for u in utterances[:5]:
    print(f"  [{u['speaker']}] {u['text'][:70]}...")

P1 - Segments: 14, Repliques: 9, Personnages: 8
P2 - Personnages castes: 8

Premieres repliques:
  [Narrateur] Et ils ne font rien de mal aux gens ? demanda timidement une grosse da...
  [Cornudet] Rassurez-vous, madame, dit Cornudet avec un sourire, ils sont corrects...
  [Elisabeth Rousset] Mon Dieu ! murmura Boule de Suif, je n'ai rien a manger. Je suis parti...
  [Comtesse de Breville] Prenez ceci, ma chere demoiselle, dit la comtesse en tendant un morcea...
  [Narrateur] Merci, madame, vous etes bien bonne, repondit Elisabeth en rougissant....


## 4. Moteur d'annotation prosodique

Le moteur d'annotation fonctionne en trois etapes pour chaque replique :

1. **Analyse lexicale** : Detection de mots-cles emotionnels dans le texte et les didascalies
2. **Contexte narratif** : Role du personnage, type de segment (dialogue/narration), position dans l'intrigue
3. **Selection des tags** : Choix des tags FishAudio les plus adapttes, avec positionnement dans le texte

Les tags sont inseres aux endroits strategiques du texte (debut de phrase, pauses naturelies, changements emotionnels).

In [4]:
@dataclass
class ProsodicAnnotation:
    # A single utterance with prosodic annotation
    seg_index: int
    speaker: str
    original_text: str
    annotated_text: str
    tags_applied: list = field(default_factory=list)
    emotion_profile: str = ""
    fish_preset: str = ""
    utterance_type: str = "dialogue"


def detect_emotion_tags(text: str) -> list:
    # Detect FishAudio tags from text content and stage directions
    text_lower = text.lower()
    detected = []

    for category, tag_dict in FISHAUDIO_TAGS.items():
        for tag_name, keywords in tag_dict.items():
            for kw in keywords:
                if kw in text_lower:
                    detected.append(tag_name)
                    break  # One match per tag is enough

    # Punctuation-based tags
    if "!" in text and text.count("!") >= 2:
        detected.append("shout")
    # Single exclamation: intensity handled by context, no extra tag needed
    # Question mark: default intonation, no extra tag needed
    if "..." in text:
        if "pause" not in detected:
            detected.append("pause")

    return list(set(detected))


def get_speaker_profile(speaker: str, characters: list, voice_config: dict) -> dict:
    # Get character profile from P1 data and P2 voice casting
    # Map speaker names to character IDs
    speaker_map = {
        "Elisabeth Rousset": "elisabeth_rousset",
        "Boule de Suif": "elisabeth_rousset",
        "Cornudet": "cornudet",
        "Comtesse de Breville": "comtesse",
        "Comte Hubert de Breville": "comte",
        "Narrateur": "narrateur",
        "Monsieur Loiseau": "loiseau",
        "Loiseau": "loiseau",
        "Monsieur Carre-Lamadon": "carre_lamadon",
        "Les bonnes soeurs": "soeurs",
        "Officier prussien": "officier",
    }
    char_id = speaker_map.get(speaker, "narrateur")

    # Get voice preset from P2 config
    fish_preset = "neutral"
    if char_id in voice_config.get("characters", {}):
        fish_info = voice_config["characters"][char_id]["voices"]["fishaudio_production"]
        fish_preset = fish_info["preset"]

    # Get role from P1 data
    role = "figurant"
    for c in characters:
        if c["id"] == char_id:
            role = c["role"]
            break

    return {
        "char_id": char_id,
        "role": role,
        "fish_preset": fish_preset,
    }


print("Moteur d'annotation prosodique defini.")
print(f"Taxonomie : {sum(len(v) for v in FISHAUDIO_TAGS.values())} cles emotionnelles")

Moteur d'annotation prosodique defini.
Taxonomie : 21 cles emotionnelles


In [5]:
def annotate_utterance(utterance: dict, characters: list, voice_config: dict) -> ProsodicAnnotation:
    # Annotate a single utterance with FishAudio expressive tags
    text = utterance["text"]
    speaker = utterance["speaker"]
    seg_index = int(utterance.get("seg_index", utterance.get("index", 0)))
    utterance_type = utterance.get("utterance_type", "dialogue")

    # Get speaker profile
    profile = get_speaker_profile(speaker, characters, voice_config)

    # Step 1: Detect emotion tags from text
    tags = detect_emotion_tags(text)

    # Step 2: Add contextual tags based on role and segment type
    if profile["role"] == "narrateur":
        if "cold" not in tags and "warm" not in tags:
            pass  # Narrator uses default neutral tone
    elif profile["role"] == "antagoniste":
        if "cold" not in tags and "sarcastic" not in tags:
            tags.append("cold")  # Antagonists default to cold tone
    elif profile["role"] == "protagoniste":
        if not any(t in tags for t in ["warm", "indignant", "sob", "gasp"]):
            tags.append("warm")  # Protagonist default warmth

    # Step 3: Build annotated text with tags inserted at natural positions
    annotated = text
    tag_placements = []

    # Prepend primary emotion tag
    if tags:
        primary_tag = tags[0]
        annotated = f"[{primary_tag}] {annotated}"
        tag_placements.append(("start", primary_tag))

    # Insert pause tags at sentence boundaries
    if ". " in text and len(tags) > 1:
        parts = annotated.split(". ", 1)
        if len(parts) == 2:
            secondary = tags[1] if len(tags) > 1 else "pause"
            annotated = f"{parts[0]}. [{secondary}] {parts[1]}"
            tag_placements.append(("mid", secondary))

    # Determine emotion profile string
    emotion_profile = "+".join(tags[:3]) if tags else "neutral"

    return ProsodicAnnotation(
        seg_index=seg_index,
        speaker=speaker,
        original_text=text,
        annotated_text=annotated,
        tags_applied=tags,
        emotion_profile=emotion_profile,
        fish_preset=profile["fish_preset"],
        utterance_type=utterance_type,
    )


print("Fonction d'annotation definie.")

Fonction d'annotation definie.


## 5. Pipeline d'annotation complete

Application du moteur d'annotation a toutes les repliques P1, avec enrichissement contextuel (didascalies, emotions detectees, profil vocal).

In [6]:
# Run full prosodic annotation pipeline
annotations = []

for u in utterances:
    ann = annotate_utterance(u, characters, voice_config)
    annotations.append(ann)

print(f"{len(annotations)} repliques annotees")
print("\n--- Apercu des annotations ---")
for ann in annotations:
    tags_str = ", ".join(ann.tags_applied) if ann.tags_applied else "(neutral)"
    print(f"\n[{ann.speaker}] (seg {ann.seg_index}) preset={ann.fish_preset}")
    print(f"  Tags: {tags_str}")
    print(f"  Profil: {ann.emotion_profile}")
    print(f"  Texte annote: {ann.annotated_text[:90]}...")

9 repliques annotees

--- Apercu des annotations ---

[Narrateur] (seg 2) preset=narrator_male
  Tags: timid
  Profil: timid
  Texte annote: [timid] Et ils ne font rien de mal aux gens ? demanda timidement une grosse dame, la comte...

[Cornudet] (seg 3) preset=expressive_male_neutral
  Tags: onctuous, laugh
  Profil: onctuous+laugh
  Texte annote: [onctuous] Rassurez-vous, madame, dit Cornudet avec un sourire, ils sont corrects....

[Elisabeth Rousset] (seg 5) preset=expressive_female_warm
  Tags: whisper
  Profil: whisper
  Texte annote: [whisper] Mon Dieu ! murmura Boule de Suif, je n'ai rien a manger. Je suis partie sans pre...

[Comtesse de Breville] (seg 6) preset=expressive_female_cold
  Tags: (neutral)
  Profil: neutral
  Texte annote: Prenez ceci, ma chere demoiselle, dit la comtesse en tendant un morceau de pain....

[Narrateur] (seg 7) preset=narrator_male
  Tags: (neutral)
  Profil: neutral
  Texte annote: Merci, madame, vous etes bien bonne, repondit Elisabeth en rougissan

## 6. Annotation des segments narratifs

Les segments de narration et description reoivent egalement des tags prosodiques pour guider le narrateur (rythme, pauses, intensite).

In [7]:
def annotate_segment(seg: dict) -> dict:
    # Annotate a narration/description segment for narrator
    text = seg["text"]
    seg_type = seg["seg_type"]
    tags = []

    # Detect emotional content
    text_lower = text.lower()
    if any(w in text_lower for w in ["peur", "effroi", "angoisse", "horreur"]):
        tags.append("pause")
        tags.append("slow")
    if any(w in text_lower for w in ["precipit", "vite", "soudain", "brusquement"]):
        tags.append("quick")
    if any(w in text_lower for w in ["silence", "calme", "tranquil", "paisibl"]):
        tags.append("slow")
        tags.append("pause")
    if any(w in text_lower for w in ["neige", "froid", "glac", "hiver"]):
        tags.append("cold")
    if "..." in text:
        tags.append("pause")

    # Default for narration
    if not tags:
        if seg_type == "description":
            tags = ["slow"]
        else:
            tags = ["pause"]

    # Build annotated text
    primary = tags[0]
    annotated = f"[{primary}] {text}"

    return {
        "seg_index": int(seg["index"]),
        "seg_type": seg_type,
        "original_text": text,
        "annotated_text": annotated,
        "tags_applied": tags,
        "speaker": "Narrateur",
        "fish_preset": "narrator_male",
    }


# Get dialogue segment indices (already annotated as utterances)
dialogue_seg_indices = set()
for u in utterances:
    idx = int(u.get("seg_index", u.get("index", -1)))
    dialogue_seg_indices.add(idx)

# Annotate non-dialogue segments
segment_annotations = []
for seg in segments:
    seg_idx = int(seg["index"])
    if seg_idx not in dialogue_seg_indices:
        seg_ann = annotate_segment(seg)
        segment_annotations.append(seg_ann)

print(f"{len(segment_annotations)} segments narratifs/descriptifs annnotes")
for sa in segment_annotations:
    print(f"  [seg {sa['seg_index']}] ({sa['seg_type']}) tags: {', '.join(sa['tags_applied'])}")

5 segments narratifs/descriptifs annnotes
  [seg 0] (narration) tags: pause
  [seg 1] (narration) tags: pause, slow
  [seg 4] (narration) tags: cold
  [seg 8] (description) tags: slow, pause, cold
  [seg 11] (narration) tags: slow, pause


## 7. Export du livre enrichi en markdown

Le format `book.ssml.md` combine les segments narratifs et les repliques annotees en un fichier markdown unique, pret pour la generation TTS (P4). Chaque segment contient les tags FishAudio et le preset vocal du personnage.

In [8]:
def build_book_ssml(
    segments: list,
    annotations: list,
    segment_annotations: list,
    voice_config: dict,
) -> str:
    # Build enriched markdown book for TTS generation
    lines = []
    lines.append("# Boule de Suif - Livre enrichi pour TTS")
    lines.append("")
    lines.append("<!-- Metadata -->")
    lines.append("<!-- source: Boule de Suif - Guy de Maupassant -->")
    lines.append(f"<!-- epic: 1028 | pass: P3 | characters: {len(voice_config['characters'])} -->")
    lines.append(f"<!-- segments: {len(segments)} | utterances: {len(annotations)} | narration: {len(segment_annotations)} -->")
    lines.append("")
    lines.append("---")
    lines.append("")

    # Build a map of all annotations by seg_index
    utterance_map = {}
    for ann in annotations:
        utterance_map[ann.seg_index] = ann

    segment_map = {}
    for sa in segment_annotations:
        segment_map[sa["seg_index"]] = sa

    # Process segments in order
    for seg in segments:
        seg_idx = int(seg["index"])
        seg_type = seg["seg_type"]

        if seg_idx in utterance_map:
            # Dialogue utterance with prosodic annotation
            ann = utterance_map[seg_idx]
            speaker = ann.speaker
            profile = get_speaker_profile(speaker, characters, voice_config)
            char_id = profile["char_id"]
            preset = ann.fish_preset
            tags = "+".join(ann.tags_applied) if ann.tags_applied else "neutral"

            lines.append(f"## Segment {seg_idx} — Dialogue")
            lines.append("")
            lines.append(f"**Personnage** : {speaker} (`{char_id}`)  ")
            lines.append(f"**Preset vocal** : `{preset}`  ")
            lines.append(f"**Profil emotionnel** : `{tags}`")
            lines.append("")
            lines.append(f"> {ann.annotated_text}")
            lines.append("")
        elif seg_idx in segment_map:
            # Narration/description segment
            sa = segment_map[seg_idx]
            tags = "+".join(sa["tags_applied"])

            lines.append(f"## Segment {seg_idx} — {seg_type.capitalize()}")
            lines.append("")
            lines.append(f"**Narrateur** | Tags: `{tags}`")
            lines.append("")
            lines.append(f"> {sa['annotated_text']}")
            lines.append("")
        else:
            # Fallback: raw segment
            lines.append(f"## Segment {seg_idx} — {seg_type.capitalize()}")
            lines.append("")
            lines.append(f"> {seg['text']}")
            lines.append("")

        lines.append("---")
        lines.append("")

    # Append voice casting summary
    lines.append("# Casting vocal")
    lines.append("")
    lines.append("| Personnage | Role | Preset FishAudio | Voix Kokoro | Voix OpenAI |")
    lines.append("|---|---|---|---|---|")
    for char_id, info in voice_config["characters"].items():
        role = info["role"]
        fish = info["voices"]["fishaudio_production"]["preset"]
        kokoro = info["voices"]["kokoro"]["voice_id"]
        openai_v = info["voices"]["openai"]["voice_id"]
        name = info["name"]
        lines.append(f"| {name} | {role} | `{fish}` | `{kokoro}` | `{openai_v}` |")
    lines.append("")

    return "\n".join(lines)


# Build and display
book_ssml = build_book_ssml(segments, annotations, segment_annotations, voice_config)
print(f"Livre enrichi genere : {len(book_ssml)} caracteres, {book_ssml.count(chr(10))} lignes")
print("\n--- Premieres lignes ---")
for line in book_ssml.split(chr(10))[:20]:
    print(line)

Livre enrichi genere : 4747 caracteres, 151 lignes

--- Premieres lignes ---
# Boule de Suif - Livre enrichi pour TTS

<!-- Metadata -->
<!-- source: Boule de Suif - Guy de Maupassant -->
<!-- epic: 1028 | pass: P3 | characters: 8 -->
<!-- segments: 14 | utterances: 9 | narration: 5 -->

---

## Segment 0 — Narration

**Narrateur** | Tags: `pause`

> [pause] Pendant plusieurs jours, des lambeaux d'armees en deroute avaient traverse la ville. Ce n'etait point de la troupe, mais des hordes debraillees.

---

## Segment 1 — Narration

**Narrateur** | Tags: `pause+slow`


## 8. Sauvegarde des resultats

In [9]:
# Save book.ssml.md
ssml_path = OUTPUT_DIR / "book.ssml.md"
ssml_path.write_text(book_ssml, encoding="utf-8")
print(f"Sauvegarde: {ssml_path.resolve()}")

# Save annotations as JSON
annotations_json = {
    "epic": "1028",
    "pass": "P3",
    "description": "Prosodic annotation with FishAudio S2-Pro tags",
    "source_text": "Boule de Suif - Guy de Maupassant",
    "stats": {
        "total_utterances": len(annotations),
        "total_narration_segments": len(segment_annotations),
        "total_tags_applied": sum(len(a.tags_applied) for a in annotations) + sum(len(sa["tags_applied"]) for sa in segment_annotations),
        "unique_tags": list(set(t for a in annotations for t in a.tags_applied)),
    },
    "utterances": [asdict(a) for a in annotations],
    "narration_segments": segment_annotations,
}

json_path = OUTPUT_DIR / "prosodic_annotations.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(annotations_json, f, ensure_ascii=False, indent=2)
print(f"Sauvegarde: {json_path.resolve()}")

# Save per-character tag summary
char_tags = {}
for ann in annotations:
    speaker = ann.speaker
    if speaker not in char_tags:
        char_tags[speaker] = []
    char_tags[speaker].extend(ann.tags_applied)

print("\n--- Resume par personnage ---")
for speaker, tags in char_tags.items():
    from collections import Counter
    tag_counts = Counter(tags)
    top_tags = ", ".join(f"{t} ({c})" for t, c in tag_counts.most_common(5))
    print(f"  {speaker}: {top_tags}")

Sauvegarde: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\prosodic_output\book.ssml.md
Sauvegarde: D:\Dev\CoursIA\MyIA.AI.Notebooks\GenAI\Audio\04-Applications\prosodic_output\prosodic_annotations.json

--- Resume par personnage ---
  Narrateur: timid (1), shout (1), indignant (1), cold (1)
  Cornudet: onctuous (1), laugh (1)
  Elisabeth Rousset: whisper (2), indignant (1)
  Comtesse de Breville: 
  Comte Hubert de Breville: pause (1), onctuous (1)


## 9. Analyse de la distribution des tags

Visualisation de la repartition des tags expressifs par categorie et par personnage, pour verifier la coherence de l'annotation.

In [10]:
from collections import Counter

# Global tag distribution
all_tags = []
for ann in annotations:
    all_tags.extend(ann.tags_applied)
for sa in segment_annotations:
    all_tags.extend(sa["tags_applied"])

tag_counter = Counter(all_tags)
total_tag_instances = sum(tag_counter.values())

print(f"=== Distribution globale des tags ===")
print(f"Total instances: {total_tag_instances}")
print(f"Tags uniques: {len(tag_counter)}")
print()

for tag, count in tag_counter.most_common():
    pct = count / total_tag_instances * 100
    bar = "#" * int(pct)
    print(f"  {tag:15s} {count:3d} ({pct:5.1f}%) {bar}")

# Per-category analysis
print("\n=== Par categorie FishAudio ===")
for cat, tag_dict in FISHAUDIO_TAGS.items():
    cat_tags = [t for t in all_tags if t in tag_dict]
    if cat_tags:
        print(f"  {cat}: {len(cat_tags)} instances ({', '.join(set(cat_tags))})")

=== Distribution globale des tags ===
Total instances: 20
Tags uniques: 9

  pause             5 ( 25.0%) #########################
  cold              3 ( 15.0%) ###############
  slow              3 ( 15.0%) ###############
  onctuous          2 ( 10.0%) ##########
  whisper           2 ( 10.0%) ##########
  indignant         2 ( 10.0%) ##########
  timid             1 (  5.0%) #####
  laugh             1 (  5.0%) #####
  shout             1 (  5.0%) #####

=== Par categorie FishAudio ===
  emotions: 1 instances (laugh)
  intensity: 3 instances (shout, whisper)
  tone: 8 instances (timid, indignant, onctuous, cold)
  rhythm: 5 instances (pause)


## 10. Conclusion

### Livrables P3

| Fichier | Description |
|---------|-------------|
| `prosodic_output/book.ssml.md` | Livre complet avec tags FishAudio inseres |
| `prosodic_output/prosodic_annotations.json` | Annotations structurees (par replique et segment) |

### Passage a P4 (Generation TTS)

Le fichier `book.ssml.md` est pret pour la generation TTS :
- Chaque segment contient les tags `[tag1] [tag2] texte` interpretes par FishAudio S2-Pro
- Le preset vocal de chaque personnage est documente
- Les tags de rythme, emotion et intensite sont positionnes aux endroits strategiques

### Limites de cette approche

- L'annotation est basee sur des heuristiques lexicales, pas sur une analyse semantique profonde
- Un modele LLM (GPT-4, Claude) pourrait ameliorer la detection emotionnelle
- Les tags FishAudio sont simules ici (FishAudio S2-Pro n'est pas deploye en Docker)